In [2]:
"""
ML Comparison: SVM  vs  Linear Regression (Ridge)
===================================================
Features  : TF-IDF on stemmed tokens
Labels    : low / medium / high   (ordinal: 0 / 1 / 2)
Split     : 80/20 stratified

Models
------
1. SVM               — LinearSVC  (standard for text classification)
2. Linear Regression — Ridge regression on ordinal labels,
                       predictions rounded to nearest class
                       (included as requested; note: Logistic Regression
                        would be the typical linear classifier for this
                        task, but Ridge is the true linear regression model)

Metrics reported per model
--------------------------
  - Accuracy
  - Weighted F1
  - Per-class Precision / Recall / F1
  - Confusion matrix
  - Cross-validation accuracy (5-fold)
"""

import json, csv, warnings
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings("ignore")



# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 1. LOAD DATA
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
rows = list(csv.DictReader(open("pipeline_dataset.csv", encoding="utf-8")))

# Use stemmed token string as feature text
texts  = [" ".join(json.loads(r["tokens_stemmed"])) for r in rows]
labels = [r["label"] for r in rows]

LABEL_ORDER = ["low", "medium", "high"]   # ordinal order for regression
label2int   = {l: i for i, l in enumerate(LABEL_ORDER)}
int2label   = {i: l for l, i in label2int.items()}

y_cat = np.array(labels)                        # string labels  (SVM)
y_ord = np.array([label2int[l] for l in labels])  # 0/1/2        (Ridge)

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 2. TRAIN / TEST SPLIT  (stratified 80/20)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
X_train, X_test, y_train_cat, y_test_cat, y_train_ord, y_test_ord = train_test_split(
    texts, y_cat, y_ord,
    test_size=0.20, random_state=42, stratify=y_cat
)
print(f"Train: {len(X_train)} samples  |  Test: {len(X_test)} samples\n")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 3. SHARED TFIDF VECTORIZER
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
tfidf = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),      # unigrams + bigrams
    min_df=2,                # ignore tokens seen in < 2 docs
    sublinear_tf=True,       # log(tf) scaling
)
X_tr = tfidf.fit_transform(X_train)
X_te = tfidf.transform(X_test)
print(f"TF-IDF features: {X_tr.shape[1]}\n")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 4.A  MODEL 1 — LinearSVC  (SVM)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
svm = LinearSVC(C=1.0, max_iter=2000, random_state=42)
svm.fit(X_tr, y_train_cat)
svm_pred = svm.predict(X_te)

svm_acc  = accuracy_score(y_test_cat, svm_pred)
svm_f1   = f1_score(y_test_cat, svm_pred, average="weighted")
svm_report = classification_report(y_test_cat, svm_pred,
                                    target_names=LABEL_ORDER, output_dict=True)
svm_cm   = confusion_matrix(y_test_cat, svm_pred, labels=LABEL_ORDER)

# CV
svm_cv = cross_val_score(
    LinearSVC(C=1.0, max_iter=2000, random_state=42),
    X_tr, y_train_cat, cv=5, scoring="accuracy"
)

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 4.B  MODEL 2 — Ridge Regression  (Linear Regression)
#      Predict ordinal value, round to nearest class
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
ridge = Ridge(alpha=1.0)
ridge.fit(X_tr, y_train_ord)
ridge_pred_raw  = ridge.predict(X_te)
ridge_pred_ord  = np.clip(np.round(ridge_pred_raw).astype(int), 0, 2)
ridge_pred_cat  = np.array([int2label[i] for i in ridge_pred_ord])

ridge_acc  = accuracy_score(y_test_cat, ridge_pred_cat)
ridge_f1   = f1_score(y_test_cat, ridge_pred_cat, average="weighted")
ridge_report = classification_report(y_test_cat, ridge_pred_cat,
                                      target_names=LABEL_ORDER, output_dict=True)
ridge_cm   = confusion_matrix(y_test_cat, ridge_pred_cat, labels=LABEL_ORDER)

# CV (on ordinal → accuracy after rounding)
from sklearn.base import BaseEstimator, ClassifierMixin
class RidgeClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, alpha=1.0):
        self.alpha = alpha
    def fit(self, X, y):
        self.ridge_ = Ridge(alpha=self.alpha)
        self.ridge_.fit(X, y)
        return self
    def predict(self, X):
        raw = self.ridge_.predict(X)
        return np.clip(np.round(raw).astype(int), 0, 2)

ridge_cv = cross_val_score(
    RidgeClassifier(alpha=1.0),
    X_tr, y_train_ord, cv=5, scoring="accuracy"
)

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 5. PRINT COMPARISON REPORT
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
SEP = "═" * 65

def print_model_block(name, acc, f1, cv_scores, cm, report):
    print(f"\n{'─'*65}")
    print(f"  {name}")
    print(f"{'─'*65}")
    print(f"  Accuracy          : {acc*100:.2f}%")
    print(f"  Weighted F1       : {f1*100:.2f}%")
    print(f"  CV Accuracy (5-fold): {cv_scores.mean()*100:.2f}% ± {cv_scores.std()*100:.2f}%")
    print(f"\n  Per-class metrics:")
    print(f"  {'Class':8s} {'Precision':>10s} {'Recall':>8s} {'F1':>8s} {'Support':>9s}")
    for cls in LABEL_ORDER:
        r = report[cls]
        print(f"  {cls:8s} {r['precision']*100:>9.1f}% {r['recall']*100:>7.1f}% "
              f"{r['f1-score']*100:>7.1f}% {int(r['support']):>9d}")
    print(f"\n  Confusion Matrix (rows=actual, cols=predicted):")
    print(f"  {'':12s} " + "  ".join(f"{l:>8s}" for l in LABEL_ORDER))
    for i, row_label in enumerate(LABEL_ORDER):
        print(f"  {row_label:12s} " + "  ".join(f"{cm[i,j]:>8d}" for j in range(3)))

print(f"\n{SEP}")
print("  ML COMPARISON: SVM  vs  LINEAR REGRESSION (Ridge)")
print(f"{SEP}")
print(f"  Dataset : 705 samples  |  Features: TF-IDF (1-2 grams)")
print(f"  Split   : 80% train ({len(X_train)})  /  20% test ({len(X_test)})")
print(f"  Classes : low / medium / high  (balanced ~33% each)")
print(f"  TF-IDF features : {X_tr.shape[1]}")

print_model_block("LinearSVC (SVM)",
                  svm_acc, svm_f1, svm_cv, svm_cm, svm_report)
print_model_block("Ridge Regression (Linear Regression, ordinal)",
                  ridge_acc, ridge_f1, ridge_cv, ridge_cm, ridge_report)

# ─── Head-to-head summary ────────────────────────────────────
print(f"\n{SEP}")
print("  HEAD-TO-HEAD SUMMARY")
print(f"{SEP}")
print(f"  {'Metric':30s} {'SVM':>10s} {'Ridge':>10s} {'Winner':>10s}")
print(f"  {'─'*60}")

metrics = [
    ("Test Accuracy",     svm_acc,       ridge_acc),
    ("Weighted F1",       svm_f1,        ridge_f1),
    ("CV Mean Accuracy",  svm_cv.mean(), ridge_cv.mean()),
    ("CV Std (lower=better)", -svm_cv.std(), -ridge_cv.std()),
]
for label, sv, ri in metrics:
    winner = "SVM ✅" if sv >= ri else "Ridge ✅"
    # flip for std row
    sv_disp = f"{abs(sv)*100:.2f}%"
    ri_disp = f"{abs(ri)*100:.2f}%"
    print(f"  {label:30s} {sv_disp:>10s} {ri_disp:>10s} {winner:>10s}")

print(f"\n{SEP}")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 6. TOP FEATURES PER CLASS (SVM coefficients)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print("\n  TOP 12 TF-IDF FEATURES PER CLASS  (SVM coefficients)")
print(f"{'─'*65}")
feature_names = np.array(tfidf.get_feature_names_out())
for i, cls in enumerate(svm.classes_):
    coefs = svm.coef_[i]
    top_idx = np.argsort(coefs)[-12:][::-1]
    print(f"\n  [{cls}]")
    for idx in top_idx:
        print(f"    {feature_names[idx]:20s}  coef={coefs[idx]:.4f}")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 7. SAVE RESULTS CSV
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
results = []
for i, (txt, true) in enumerate(zip(X_test, y_test_cat)):
    results.append({
        "text":           txt[:80],
        "true_label":     true,
        "svm_pred":       svm_pred[i],
        "svm_correct":    int(svm_pred[i] == true),
        "ridge_pred":     ridge_pred_cat[i],
        "ridge_correct":  int(ridge_pred_cat[i] == true),
    })

with open("ml_predictions.csv", "w", encoding="utf-8", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(results[0].keys()))
    w.writeheader(); w.writerows(results)

# Save score summary JSON
summary = {
    "svm":   {"accuracy": round(svm_acc,4),   "weighted_f1": round(svm_f1,4),
              "cv_mean": round(float(svm_cv.mean()),4),  "cv_std": round(float(svm_cv.std()),4)},
    "ridge": {"accuracy": round(ridge_acc,4), "weighted_f1": round(ridge_f1,4),
              "cv_mean": round(float(ridge_cv.mean()),4),"cv_std": round(float(ridge_cv.std()),4)},
}
with open("ml_scores.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(f"\n✅  ml_predictions.csv  and  ml_scores.json  saved.")

Train: 564 samples  |  Test: 141 samples

TF-IDF features: 768


═════════════════════════════════════════════════════════════════
  ML COMPARISON: SVM  vs  LINEAR REGRESSION (Ridge)
═════════════════════════════════════════════════════════════════
  Dataset : 705 samples  |  Features: TF-IDF (1-2 grams)
  Split   : 80% train (564)  /  20% test (141)
  Classes : low / medium / high  (balanced ~33% each)
  TF-IDF features : 768

─────────────────────────────────────────────────────────────────
  LinearSVC (SVM)
─────────────────────────────────────────────────────────────────
  Accuracy          : 74.47%
  Weighted F1       : 74.42%
  CV Accuracy (5-fold): 70.04% ± 2.51%

  Per-class metrics:
  Class     Precision   Recall       F1   Support
  low           83.7%    85.4%    84.5%        48
  medium        68.2%    68.2%    68.2%        44
  high          70.8%    69.4%    70.1%        49

  Confusion Matrix (rows=actual, cols=predicted):
                    low    medium      high
  lo